# Top-K Token Pruning Testing & Visualization (ImageNet-100)

This notebook tests the Top-K token pruning functionality and provides visualizations of:
- Which patches are removed at each reduction layer
- Attention patterns from CLS token
- Token reduction statistics
- Performance vs. accuracy trade-offs

In [ ]:
import getpass

token = getpass.getpass("Enter GitHub token: ")

!git clone https://{token}@github.com/Chalhotra/ViT-Token-Economy.git
%cd ViT-Token-Economy

In [ ]:
!git checkout test-branch

In [ ]:
!pip -q install -r requirements.txt
!pip -q install -e .

In [ ]:
# Import core modules
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader
from src.eval import evaluate_accuracy_latency_throughput, compute_gflops
from src.utils import get_device, num_params
from src.test_models.topk import TopKConfig, apply_topk_pruning, AttentionTopKFromExisting
import torch
import torch.nn as nn

# Import visualization libraries
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import seaborn as sns
from typing import List, Tuple, Dict
import pandas as pd

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
device = get_device()
maps = build_imagenet100_to_1k_map()
print(f"Using device: {device}")

## Utility Functions for Visualization

In [ ]:
def extract_attention_maps(model, x):
    """
    Extract attention maps from all blocks in the model.
    Returns list of attention tensors and pruning indices.
    """
    attention_maps = []
    pruning_indices = []
    token_counts = []
    
    # Get initial token count
    B, N, C = x.shape
    token_counts.append(N - 1)  # exclude CLS token
    
    # Forward through blocks and collect attention
    for i, block in enumerate(model.blocks):
        if hasattr(block, 'attn'):
            # Check if this is a TopK block
            if isinstance(block.attn, AttentionTopKFromExisting):
                # Run forward to populate last_idx
                with torch.no_grad():
                    x = block(x)
                
                # Get stored indices
                idx = block.attn.last_idx
                if idx is not None:
                    pruning_indices.append((i, idx.cpu()))
                    token_counts.append(idx.shape[1])
                else:
                    token_counts.append(token_counts[-1])
            else:
                with torch.no_grad():
                    x = block(x)
                token_counts.append(token_counts[-1])
        else:
            with torch.no_grad():
                x = block(x)
            token_counts.append(token_counts[-1])
    
    return pruning_indices, token_counts


def visualize_patch_pruning(image, patch_size, pruning_indices, token_counts, model_name):
    """
    Visualize which patches are kept/removed at each pruning layer.
    
    Args:
        image: Original PIL image or tensor
        patch_size: Size of patches (e.g., 16)
        pruning_indices: List of (layer_idx, kept_indices_tensor)
        token_counts: List of token counts at each layer
        model_name: Name for the plot title
    """
    # Convert image to numpy if needed
    if isinstance(image, torch.Tensor):
        img_np = image.permute(1, 2, 0).cpu().numpy()
    else:
        img_np = np.array(image)
    
    # Normalize if needed
    if img_np.max() > 1.0:
        img_np = img_np / 255.0
    
    h, w = img_np.shape[:2]
    n_patches_h = h // patch_size
    n_patches_w = w // patch_size
    total_patches = n_patches_h * n_patches_w
    
    # Create visualization
    n_pruning_layers = len(pruning_indices)
    fig, axes = plt.subplots(1, n_pruning_layers + 1, figsize=(5 * (n_pruning_layers + 1), 5))
    if n_pruning_layers == 0:
        axes = [axes]
    
    # Show original image
    axes[0].imshow(img_np)
    axes[0].set_title(f'Original Image\n{total_patches} patches')
    axes[0].axis('off')
    
    # Show pruned versions
    for idx, (layer_idx, kept_indices) in enumerate(pruning_indices):
        ax = axes[idx + 1]
        
        # Create mask for kept patches
        mask = np.zeros((n_patches_h, n_patches_w))
        kept_idx_np = kept_indices[0].numpy()  # Take first batch item
        
        # Convert linear indices to 2D
        for ki in kept_idx_np:
            pi = ki % n_patches_h
            pj = ki // n_patches_h
            if pi < n_patches_h and pj < n_patches_w:
                mask[pi, pj] = 1
        
        # Create overlay
        overlay = img_np.copy()
        for i in range(n_patches_h):
            for j in range(n_patches_w):
                if mask[i, j] == 0:  # Removed patch
                    y_start, y_end = i * patch_size, (i + 1) * patch_size
                    x_start, x_end = j * patch_size, (j + 1) * patch_size
                    overlay[y_start:y_end, x_start:x_end] = overlay[y_start:y_end, x_start:x_end] * 0.2
        
        kept_count = int(mask.sum())
        removed_count = total_patches - kept_count
        keep_rate = kept_count / total_patches
        
        ax.imshow(overlay)
        ax.set_title(f'After Layer {layer_idx}\n{kept_count}/{total_patches} patches ({keep_rate:.1%})\nRemoved: {removed_count}')
        ax.axis('off')
    
    plt.suptitle(f'{model_name} - Token Pruning Visualization', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_token_reduction_timeline(token_counts, reduction_locs, model_name):
    """
    Plot how token count decreases through the layers.
    """
    layers = list(range(len(token_counts)))
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(layers, token_counts, marker='o', linewidth=2, markersize=8)
    
    # Highlight reduction locations
    for loc in reduction_locs:
        if loc < len(token_counts):
            ax.axvline(x=loc, color='red', linestyle='--', alpha=0.5, label=f'Reduction at layer {loc}')
    
    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel('Number of Patch Tokens', fontsize=12)
    ax.set_title(f'{model_name} - Token Count Through Layers', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # Add percentage annotations
    initial = token_counts[0]
    for i, count in enumerate(token_counts):
        if i in reduction_locs or i == 0 or i == len(token_counts) - 1:
            pct = count / initial * 100
            ax.annotate(f'{count} ({pct:.1f}%)', 
                       xy=(i, count), 
                       xytext=(0, 10), 
                       textcoords='offset points',
                       ha='center',
                       fontsize=9,
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))
    
    plt.tight_layout()
    plt.show()


def compare_topk_configurations(results_list: List[Dict], title: str = "Top-K Configuration Comparison"):
    """
    Create comparison plots for different Top-K configurations.
    
    Args:
        results_list: List of result dicts with keys: config_name, accuracy, gflops, params_m, latency, throughput
    """
    df = pd.DataFrame(results_list)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Accuracy comparison
    axes[0, 0].bar(range(len(df)), df['top1_acc'], color='steelblue')
    axes[0, 0].set_xticks(range(len(df)))
    axes[0, 0].set_xticklabels(df['config_name'], rotation=45, ha='right')
    axes[0, 0].set_ylabel('Top-1 Accuracy (%)')
    axes[0, 0].set_title('Accuracy Comparison')
    axes[0, 0].grid(True, alpha=0.3)
    
    # GFLOPs comparison
    axes[0, 1].bar(range(len(df)), df['gflops'], color='coral')
    axes[0, 1].set_xticks(range(len(df)))
    axes[0, 1].set_xticklabels(df['config_name'], rotation=45, ha='right')
    axes[0, 1].set_ylabel('GFLOPs')
    axes[0, 1].set_title('Computational Cost')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Latency comparison
    axes[1, 0].bar(range(len(df)), df['latency_ms'], color='mediumseagreen')
    axes[1, 0].set_xticks(range(len(df)))
    axes[1, 0].set_xticklabels(df['config_name'], rotation=45, ha='right')
    axes[1, 0].set_ylabel('Latency (ms)')
    axes[1, 0].set_title('Inference Latency')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Efficiency plot (Accuracy vs GFLOPs)
    axes[1, 1].scatter(df['gflops'], df['top1_acc'], s=100, alpha=0.6, c=range(len(df)), cmap='viridis')
    for i, row in df.iterrows():
        axes[1, 1].annotate(row['config_name'], 
                           (row['gflops'], row['top1_acc']),
                           xytext=(5, 5), 
                           textcoords='offset points',
                           fontsize=8)
    axes[1, 1].set_xlabel('GFLOPs')
    axes[1, 1].set_ylabel('Top-1 Accuracy (%)')
    axes[1, 1].set_title('Efficiency Plot')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Print summary table
    print("\n" + "="*80)
    print(f"{title} - Summary Table")
    print("="*80)
    print(df.to_string(index=False))
    print("="*80 + "\n")

## Test Top-K with Different Configurations

In [ ]:
def run_topk_test(
    model_id: str,
    topk_config: TopKConfig,
    config_name: str,
    batch_size: int = 64,
    visualize: bool = True
):
    """
    Run model with specific Top-K configuration and optionally visualize.
    """
    print(f"\n{'='*80}")
    print(f"Testing: {config_name}")
    print(f"Model: {model_id}")
    print(f"Reduction locations: {topk_config.reduction_loc}")
    print(f"Keep rates: {topk_config.keep_rate}")
    print(f"{'='*80}\n")
    
    # Create model
    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    
    # Apply Top-K pruning
    model = apply_topk_pruning(model, topk_config)
    model = model.to(device).eval()
    
    # Load data
    ds = load_imagenet100_split(DataConfig(split='validation'))
    transform = build_transform_for_model(model)
    ds_t = apply_timm_preprocess(ds, transform)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))
    
    # Evaluate
    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    
    # Compute GFLOPs
    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)
    
    # Visualization
    if visualize and topk_config.enabled:
        # Get a sample image for visualization
        sample_idx = 42  # arbitrary sample
        sample_image = ds[sample_idx]['image']
        sample_tensor = ds_t[sample_idx]['pixel_values'].unsqueeze(0).to(device)
        
        # Extract pruning information
        with torch.no_grad():
            # Embed patches
            x = model.patch_embed(sample_tensor)
            if hasattr(model, 'cls_token'):
                cls_tokens = model.cls_token.expand(sample_tensor.shape[0], -1, -1)
                x = torch.cat((cls_tokens, x), dim=1)
            if hasattr(model, 'pos_embed'):
                x = x + model.pos_embed
            if hasattr(model, 'pos_drop'):
                x = model.pos_drop(x)
            
            # Extract attention and pruning info
            pruning_indices, token_counts = extract_attention_maps(model, x)
        
        # Visualize patch pruning
        if pruning_indices:
            visualize_patch_pruning(
                sample_image, 
                16,  # patch size
                pruning_indices, 
                token_counts,
                f"{model_id} - {config_name}"
            )
            
            # Plot token reduction timeline
            plot_token_reduction_timeline(
                token_counts,
                list(topk_config.reduction_loc),
                f"{model_id} - {config_name}"
            )
    
    result = {
        'config_name': config_name,
        'model': model_id,
        'params_m': num_params(model) / 1e6,
        'gflops': gflops,
        **metrics
    }
    
    print(f"\nResults for {config_name}:")
    print(f"  Top-1 Accuracy: {metrics['top1_acc']:.2f}%")
    print(f"  Top-5 Accuracy: {metrics['top5_acc']:.2f}%")
    print(f"  GFLOPs: {gflops:.3f}")
    print(f"  Latency: {metrics['latency_ms']:.2f} ms")
    print(f"  Throughput: {metrics['throughput_samples_per_sec']:.1f} samples/sec")
    
    return result

## Baseline (No Pruning)

In [ ]:
# Test baseline without Top-K
baseline_config = TopKConfig(enabled=False)
baseline_result = run_topk_test(
    model_id='deit_tiny_patch16_224',
    topk_config=baseline_config,
    config_name='Baseline (No Pruning)',
    visualize=False
)

## Sanity Check: Keep Rate = 1.0 (Should Match Baseline)

In [ ]:
# Sanity check: keep_rate = 1.0 should yield same accuracy
sanity_config = TopKConfig(
    enabled=True,
    keep_rate=(1.0,),
    reduction_loc=(3, 6, 9),
    exponentiate_single_keep_rate=False
)
sanity_result = run_topk_test(
    model_id='deit_tiny_patch16_224',
    topk_config=sanity_config,
    config_name='Sanity Check (keep=1.0)',
    visualize=True
)

## Test Different Keep Rates (Single Value, Exponentiated)

In [ ]:
# Test with keep_rate = 0.9 (will be exponentiated: 0.9, 0.81, 0.729)
results = [baseline_result, sanity_result]

keep_09_config = TopKConfig(
    enabled=True,
    keep_rate=(0.9,),
    reduction_loc=(3, 6, 9),
    exponentiate_single_keep_rate=True
)
result_09 = run_topk_test(
    model_id='deit_tiny_patch16_224',
    topk_config=keep_09_config,
    config_name='TopK keep=0.9^i',
    visualize=True
)
results.append(result_09)

In [ ]:
# Test with keep_rate = 0.8 (will be exponentiated: 0.8, 0.64, 0.512)
keep_08_config = TopKConfig(
    enabled=True,
    keep_rate=(0.8,),
    reduction_loc=(3, 6, 9),
    exponentiate_single_keep_rate=True
)
result_08 = run_topk_test(
    model_id='deit_tiny_patch16_224',
    topk_config=keep_08_config,
    config_name='TopK keep=0.8^i',
    visualize=True
)
results.append(result_08)

In [ ]:
# Test with keep_rate = 0.7 (will be exponentiated: 0.7, 0.49, 0.343)
keep_07_config = TopKConfig(
    enabled=True,
    keep_rate=(0.7,),
    reduction_loc=(3, 6, 9),
    exponentiate_single_keep_rate=True
)
result_07 = run_topk_test(
    model_id='deit_tiny_patch16_224',
    topk_config=keep_07_config,
    config_name='TopK keep=0.7^i',
    visualize=True
)
results.append(result_07)

## Test Custom Keep Rates (Multiple Values)

In [ ]:
# Test with custom keep rates (no exponentiation)
custom_config = TopKConfig(
    enabled=True,
    keep_rate=(0.9, 0.8, 0.7),
    reduction_loc=(3, 6, 9),
    exponentiate_single_keep_rate=False
)
result_custom = run_topk_test(
    model_id='deit_tiny_patch16_224',
    topk_config=custom_config,
    config_name='TopK keep=[0.9,0.8,0.7]',
    visualize=True
)
results.append(result_custom)

## Test Different Reduction Locations

In [ ]:
# Early pruning: layers 2, 4
early_config = TopKConfig(
    enabled=True,
    keep_rate=(0.85, 0.7),
    reduction_loc=(2, 4),
    exponentiate_single_keep_rate=False
)
result_early = run_topk_test(
    model_id='deit_tiny_patch16_224',
    topk_config=early_config,
    config_name='Early Pruning (2,4)',
    visualize=True
)
results.append(result_early)

In [ ]:
# Late pruning: layers 8, 10
late_config = TopKConfig(
    enabled=True,
    keep_rate=(0.85, 0.7),
    reduction_loc=(8, 10),
    exponentiate_single_keep_rate=False
)
result_late = run_topk_test(
    model_id='deit_tiny_patch16_224',
    topk_config=late_config,
    config_name='Late Pruning (8,10)',
    visualize=True
)
results.append(result_late)

## Aggressive Pruning Test

In [ ]:
# Aggressive pruning
aggressive_config = TopKConfig(
    enabled=True,
    keep_rate=(0.6,),
    reduction_loc=(3, 6, 9),
    exponentiate_single_keep_rate=True
)
result_aggressive = run_topk_test(
    model_id='deit_tiny_patch16_224',
    topk_config=aggressive_config,
    config_name='Aggressive keep=0.6^i',
    visualize=True
)
results.append(result_aggressive)

## Comparison Across All Configurations

In [ ]:
# Compare all configurations
compare_topk_configurations(results, title="DeiT-Tiny Top-K Pruning Comparison")

## Test on ViT-Tiny

In [ ]:
# Run similar tests on ViT-Tiny
vit_results = []

# Baseline
vit_baseline = run_topk_test(
    model_id='vit_tiny_patch16_224',
    topk_config=TopKConfig(enabled=False),
    config_name='ViT Baseline',
    visualize=False
)
vit_results.append(vit_baseline)

# With moderate pruning
vit_moderate = run_topk_test(
    model_id='vit_tiny_patch16_224',
    topk_config=TopKConfig(enabled=True, keep_rate=(0.8,), reduction_loc=(3, 6, 9)),
    config_name='ViT TopK 0.8^i',
    visualize=True
)
vit_results.append(vit_moderate)

# Compare
compare_topk_configurations(vit_results, title="ViT-Tiny Top-K Pruning Comparison")

## Summary and Conclusions

In [ ]:
# Create a comprehensive summary
print("\n" + "="*100)
print("COMPREHENSIVE SUMMARY - ALL TESTS")
print("="*100)

all_results = results + vit_results
summary_df = pd.DataFrame(all_results)

# Calculate efficiency metrics
summary_df['acc_per_gflop'] = summary_df['top1_acc'] / summary_df['gflops']
summary_df['acc_drop_from_baseline'] = summary_df['top1_acc'] - summary_df['top1_acc'].iloc[0]
summary_df['gflops_reduction_%'] = (1 - summary_df['gflops'] / summary_df['gflops'].iloc[0]) * 100

print("\nFull Results Table:")
print(summary_df.to_string(index=False))

print("\n" + "="*100)
print("KEY INSIGHTS:")
print("="*100)

# Find best efficiency
best_efficiency_idx = summary_df['acc_per_gflop'].idxmax()
print(f"Best Accuracy/GFLOPs: {summary_df.loc[best_efficiency_idx, 'config_name']}")
print(f"  - Accuracy: {summary_df.loc[best_efficiency_idx, 'top1_acc']:.2f}%")
print(f"  - GFLOPs: {summary_df.loc[best_efficiency_idx, 'gflops']:.3f}")
print(f"  - Efficiency: {summary_df.loc[best_efficiency_idx, 'acc_per_gflop']:.3f}")

# Find maximum GFLOPs reduction with minimal accuracy drop
reasonable_configs = summary_df[summary_df['acc_drop_from_baseline'] > -5.0]  # Less than 5% drop
if len(reasonable_configs) > 1:
    best_reduction_idx = reasonable_configs['gflops_reduction_%'].idxmax()
    print(f"\nBest GFLOPs Reduction (with <5% acc drop): {reasonable_configs.loc[best_reduction_idx, 'config_name']}")
    print(f"  - Accuracy: {reasonable_configs.loc[best_reduction_idx, 'top1_acc']:.2f}%")
    print(f"  - Accuracy Drop: {reasonable_configs.loc[best_reduction_idx, 'acc_drop_from_baseline']:.2f}%")
    print(f"  - GFLOPs Reduction: {reasonable_configs.loc[best_reduction_idx, 'gflops_reduction_%']:.1f}%")

print("\n" + "="*100)